# Togo — EDA, Cleaning & Visuals
This notebook performs data profiling, outlier detection (Z-score), median imputation, exports a cleaned CSV to `../data/togo_clean.csv`, and includes key plots required by Task 2.

In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

DATA_DIR = Path('..') / 'data'
INFILE = DATA_DIR / 'togo-dapaong_qc.csv'
OUTFILE = DATA_DIR / 'togo_clean.csv'

df = pd.read_csv(INFILE)
print('Loaded', INFILE.name, 'shape=', df.shape)

def norm(s):
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

cols_norm = {norm(c): c for c in df.columns}
candidates = ['ghi','dni','dhi','moda','modb','ws','wsgust','tamb','rh','timestamp']
found = {cand: next((orig for key, orig in cols_norm.items() if key.startswith(cand) or cand.startswith(key)), None) for cand in candidates}
print('Found mapping:')
for k,v in found.items():
    print(f'  {k:8s} -> {v}')

In [ ]:
if found.get('timestamp'):
    tcol = found['timestamp']
    df[tcol] = pd.to_datetime(df[tcol], errors='coerce')
    df = df.sort_values(by=tcol)
else:
    tcol = None
numeric_candidates = ['ghi','dni','dhi','moda','modb','ws','wsgust']
numeric_cols = [found[c] for c in numeric_candidates if found.get(c) is not None]
numeric_cols = [c for c in numeric_cols if c is not None]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')
print('Numeric columns used:', numeric_cols)
display(df.describe(include='number').T)
print('
Missing values per column:')
print((df.isna().sum()).sort_values(ascending=False)[:10])

In [ ]:
outlier_flags = pd.DataFrame(index=df.index)
for c in numeric_cols:
    mean = df[c].mean()
    std = df[c].std(ddof=0)
    if pd.isna(std) or std == 0:
        outlier_flags[c] = False
        continue
    z = (df[c] - mean)/std
    outlier_flags[c] = z.abs() > 3
    print(f'Outliers in {c}:', int(outlier_flags[c].sum()))
any_outlier = outlier_flags.any(axis=1) if not outlier_flags.empty else pd.Series(False, index=df.index)
print('Total rows with any outlier:', int(any_outlier.sum()))
df_clean = df.loc[~any_outlier].copy()
print('After dropping outliers shape=', df_clean.shape)
for c in numeric_cols:
    med = df_clean[c].median(skipna=True)
    df_clean[c] = df_clean[c].fillna(med)
    print(f'Filled NA in {c} with median={med}')
df_clean.to_csv(OUTFILE, index=False)
print('Wrote cleaned CSV to', OUTFILE)

## Plots: Time series, histogram, and correlation heatmap

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
fig, ax = plt.subplots(figsize=(12,4))
if tcol and found.get('ghi'):
    sns.lineplot(data=df_clean, x=tcol, y=found['ghi'], ax=ax, lw=0.6)
    ax.set_title('GHI time series (Togo)')
else:
    ax.text(0.5,0.5,'No timestamp or GHI column found', ha='center')
plt.show()

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,4))
if found.get('ghi'):
    sns.histplot(df_clean[found['ghi']].dropna(), bins=60, kde=True, ax=axes[0])
    axes[0].set_title('GHI distribution (Togo)')
    sns.boxplot(x=df_clean[found['ghi']], ax=axes[1])
    axes[1].set_title('GHI boxplot (Togo)')
else:
    axes[0].text(0.5,0.5,'No GHI column', ha='center')
    axes[1].text(0.5,0.5,'No GHI column', ha='center')
plt.show()

In [ ]:
corr_cols = [c for c in numeric_cols if c in df_clean.columns]
if corr_cols:
    corr = df_clean[corr_cols].corr()
    fig, ax = plt.subplots(figsize=(8,6))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', ax=ax)
    ax.set_title('Correlation heatmap')
    plt.show()
else:
    print('No numeric columns available for correlation')

In [ ]:
# Wind rose (polar) — uses wind direction and wind speed if present
wd_candidates = ['wd','winddir','winddirection','wdg','wind_dir']
wd_col = next((col for col in df_clean.columns if any(col.lower().startswith(c) for c in wd_candidates)), None)
ws_col = found.get('ws') or next((col for col in df_clean.columns if col.lower().startswith('ws')), None)
if wd_col and ws_col:
    import numpy as np
    dirs = pd.to_numeric(df_clean[wd_col], errors='coerce').dropna() % 360
    speeds = pd.to_numeric(df_clean.loc[dirs.index, ws_col], errors='coerce').fillna(0)
    bins = np.arange(0, 360 + 30, 30)
    labels = (bins[:-1] + bins[1:]) / 2
    dir_bin = pd.cut(dirs, bins=bins, include_lowest=True, right=False)
    rose = speeds.groupby(dir_bin).mean().reindex(pd.IntervalIndex.from_breaks(bins, closed='left'))
    theta = np.deg2rad(labels)
    values = rose.values
    fig, ax = plt.subplots(figsize=(6,6), subplot_kw=dict(polar=True))
    width = np.deg2rad(30)
    ax.bar(theta, values, width=width, bottom=0.0, align='center')
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_title('Wind rose — mean WS by direction')
    plt.show()
else:
    print('Wind rose requires wind-direction and wind-speed columns — not found.')

In [ ]:
# Bubble chart: GHI vs Tamb with bubble size = RH (if available)
ghi_col = found.get('ghi')
tamb_col = found.get('tamb')
rh_col = found.get('rh')
if ghi_col and tamb_col:
    if rh_col and rh_col in df_clean.columns:
        sizes = (df_clean[rh_col].fillna(df_clean[rh_col].median()) - df_clean[rh_col].min() + 1) * 5
    else:
        sizes = None
    plt.figure(figsize=(8,6))
    sns.scatterplot(x=tamb_col, y=ghi_col, size=sizes, sizes=(20,300), data=df_clean, alpha=0.5, legend=False)
    plt.xlabel('Tamb')
    plt.ylabel('GHI')
    plt.title('GHI vs Tamb (bubble size = RH)')
    plt.show()
else:
    print('Bubble chart requires GHI and Tamb columns — not found.')